# Учебный блокнот для главы 17
**Создайте свою первую LLM — Глава 17: Варианты развертывания**

Откройте блокнот в Jupyter или загрузите его в Colab. Ячейки только печатают примеры файлов и команды для терминала. Они ничего не устанавливают, не требуют учётных данных, не отправляют запросы и не запускают GPU.

Если решите выполнить напечатанные команды, используйте свой локальный терминал. За работу удалённых функций может взиматься плата.

In [ ]:
# Необязательная настройка в локальном терминале, не в блокноте.
print('В виртуальном окружении с Python 3.11:')
print('python -m pip install modal fastapi pydantic requests')
print('modal setup  # Вход для CLI; это не прокси-токен.')

## Раздел 17.1: Зачем нужен Modal?

**Что такое бессерверные вычисления (Serverless)?**

| Традиционный сервер | Бессерверные (Modal) |
|---------------------|----------------------|
| Вы арендуете компьютер 24/7 | Код выполняется только при необходимости |
| Вы платите даже когда он простаивает | Вы платите за секунды вычислений |
| Вы управляете сервером | Modal управляет сервером, вы — доступом к приложению |
| Фиксированная мощность | Автоматическое масштабирование |

**Аналогия:** Традиционный хостинг похож на владение автомобилем. Бессерверные вычисления (Serverless) похожи на использование такси, вы платите только когда едете.

**Зачем использовать Modal для LLM?**
- Встроенная поддержка GPU (от T4 до H100)
- $30/месяц бесплатных кредитов (~37 часов GPU L4; ~50 часов на T4)
- Встроенная поддержка Python (без Docker, без YAML)
- Масштабирование до нуля: плата за вычисления на GPU прекращается после остановки контейнеров

## Раздел 17.2: Привет, Modal

Простейшее приложение Modal. Сохраните это как `hello.py` и запустите с помощью `modal run hello.py`:

In [ ]:
# Сохраните это как hello.py
hello_modal_code = '''
import modal

app = modal.App("hello-world")

@app.function()
def hello(name: str) -> str:
    return f"Hello, {name}!"

@app.local_entrypoint()
def main():
    result = hello.remote("World")
    print(result)
'''

print("Сохраните этот код как 'hello.py':")
print(hello_modal_code)
print("\nЗатем выполните: modal run hello.py")

**Если выполнить `modal run hello.py` в терминале:**

1. `modal.App()` создает ваше приложение
2. `@app.function()` помечает код для выполнения в облаке Modal
3. `hello.remote()` вызывает функцию удаленно
4. Modal запустил контейнер, выполнил ваш код, вернул результат

## Раздел 17.3: Добавление поддержки GPU

Один параметр добавляет поддержку GPU:

In [ ]:
# Сохраните это как gpu_test.py
gpu_test_code = '''
import modal

app = modal.App("gpu-test")

# torch должен быть установлен в образ контейнера (следующий раздел — об образах)
image = modal.Image.debian_slim(python_version="3.11").pip_install("torch")

@app.function(image=image, gpu="L4")  # Вот и всё!
def check_gpu():
    import torch
    if torch.cuda.is_available():
        device = torch.cuda.get_device_name(0)
        return f"GPU available: {device}"
    return "No GPU found"

@app.local_entrypoint()
def main():
    print(check_gpu.remote())
'''

print("Сохраните этот код как 'gpu_test.py':")
print(gpu_test_code)
print("\nЗатем выполните: modal run gpu_test.py")
print("\nОжидаемый вывод: GPU available: NVIDIA L4")

**Один параметр.** Без установки CUDA, без управления драйверами, без образов Docker. Modal управляет всем стеком GPU.

## Раздел 17.4: Установка зависимостей

Вашей LLM нужны библиотеки. Modal позволяет определить окружение контейнера на Python:

In [ ]:
# Сохраните это как with_deps.py
with_deps_code = '''
import modal

# Определяем образ контейнера
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "torch",
    "transformers",
    "accelerate",
)

app = modal.App("with-deps")

@app.function(image=image, gpu="T4")
def generate_text():
    from transformers import pipeline
    
    generator = pipeline("text-generation", model="gpt2", device=0)
    result = generator("The meaning of life is", max_length=50)
    return result[0]["generated_text"]

@app.local_entrypoint()
def main():
    print(generate_text.remote())
'''

print("Сохраните этот код как 'with_deps.py':")
print(with_deps_code)
print("\nЗатем выполните: modal run with_deps.py")

Параметр `image` сообщает Modal, что нужно установить. Первый запуск занимает больше времени (построение образа), но последующие запуски используют кешированный образ.

## Раздел 17.5: Веб-конечные точки с FastAPI

Этот пример работает на CPU: `/health` возвращает статический статус, а `/hello/{name}` — приветствие. Прокси Modal защищает всё ASGI-приложение, включая документацию. Знание URL не даёт права доступа.

Для обоих веб-примеров создайте прокси-токен в Modal Settings → Proxy auth tokens или командой `modal workspace proxy-tokens create --name chapter17` в приватном терминале. Сохраните ID `wk-…` и однократно показываемый секрет `ws-…` в менеджере паролей. Выделите среду для примера; если включён ролевой доступ, ограничьте токен этой средой. Имя токена не ограничивает его действие одним приложением. Не помещайте пару в блокнот, браузерный клиент, снимки экрана или репозиторий.

In [ ]:
# Сохраните это как simple_api.py
simple_api_code = '''
import modal
from fastapi import FastAPI

app = modal.App("my-api")
web_app = FastAPI()

# Зависимости нужны в образе контейнера, а не только на вашем компьютере.
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi==0.163.2",
    "uvicorn==0.54.0",
    "httpx==0.28.1",
    "pydantic==2.14.3",
)

@web_app.get("/health")
def health():
    return {"status": "healthy"}

@web_app.get("/hello/{name}")
def hello(name: str):
    return {"message": f"Hello, {name}!"}

@app.function(image=image)
@modal.asgi_app(requires_proxy_auth=True)
def serve():
    return web_app
'''

print("Сохраните этот код как 'simple_api.py':")
print(simple_api_code)
print("\nЗатем выполните: modal deploy simple_api.py")
print("\nВы получите URL вида: https://your-workspace--my-api-serve.modal.run")

**Необязательная проверка в терминале.** Задайте `SIMPLE_URL` в приватном сеансе Bash: это точный HTTPS-адрес приложения **my-api** из вывода вашего развёртывания. Загрузите `MODAL_KEY` и `MODAL_SECRET` из менеджера секретов в окружение. Не вводите значения в команды, сохраняемые в истории, и не включайте трассировку оболочки.

```bash
curl -i --max-time 30 "$SIMPLE_URL/health"  # Ожидается 401 от прокси.
curl -i --max-time 30 --header @- "$SIMPLE_URL/health" <<EOF
Modal-Key: $MODAL_KEY
Modal-Secret: $MODAL_SECRET
EOF
# Ожидается {"status":"healthy"}. Для /hello/World нужны те же заголовки.
modal app stop my-api
```

Даже если проверка не прошла, остановите `my-api` и убедитесь в панели управления, что контейнеров не осталось. Обычная вкладка браузера не передаёт прокси-пару: прокси отклонит запрос до запуска функции.

## Раздел 17.6: Полноценный сервис LLM

Шаблон рассчитан на предобученную Qwen3.5-2B на L4. Прокси Modal проверяет пару `Modal-Key`/`Modal-Secret` до запуска или пробуждения контейнера. `/health` проверяет готовность vLLM без генерации. Перед генерацией `/chat` отдельно проверяет bearer-токен приложения внутри FastAPI. Даже если `/chat` затем вернёт 401, клиент с действующей прокси-парой уже может запустить платный GPU.

In [ ]:
# Сохраните это как llm_service.py
# Печатаем исходный код. Локальный vLLM не должен быть доступен клиентам напрямую.
llm_service_code = '''
import modal
from fastapi import FastAPI, HTTPException, Request, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.responses import JSONResponse
from pydantic import BaseModel, field_validator
import os
from collections import defaultdict
import threading
import uuid
from datetime import datetime
import subprocess
import time
import requests

MODEL = "Qwen/Qwen3.5-2B"
# vLLM поддерживает Qwen3.5 напрямую: trust_remote_code не нужен.
# Этот флаг разрешает исполнение кода из репозитория модели; сначала изучите код.

image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi",
    "vllm==0.34.0",  # Фиксируем версию для воспроизводимости.
    "torch",
    "transformers",
    "requests",
)

app = modal.App("my-llm-service")
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)
web_app = FastAPI(title="My LLM API", version="1.0.0")

# Схема bearer добавляет кнопку Authorize в Swagger, но сам токен проверяет require_token.
bearer_scheme = HTTPBearer(auto_error=False)

# Ограничения из главы 16, авторизация и бюджет токенов
MAX_PROMPT_CHARS = 4000  # Быстрый первый фильтр; окончательная проверка — по токенам.
API_TOKEN = os.environ.get("API_TOKEN", "")  # Значение поступает из секрета Modal.
MAX_INPUT_TOKENS = 2048  # Учитываем шаблон чата; вход и выход вместе помещаются в 8192 токена.
MAX_OUTPUT_TOKENS = 512  # предел генерации для одного запроса, а не лимит расходов
RATE_LIMIT = 20
RATE_WINDOW = 60  # секунды по монотонным часам
request_counts: dict[str, list[float]] = defaultdict(list)
_rate_lock = threading.Lock()

def require_token(credentials: HTTPAuthorizationCredentials | None = Depends(bearer_scheme)):
    """Аутентификация Bearer-токеном: общий GPU-эндпоинт не должен быть открыт всему миру."""
    if not API_TOKEN or credentials is None or credentials.credentials != API_TOKEN:
        raise HTTPException(401, "Missing or invalid token")

class ChatRequest(BaseModel):
    message: str
    max_tokens: int = 256

    @field_validator("message")
    @classmethod
    def not_empty(cls, v):
        if not v.strip():
            raise ValueError("Message cannot be empty")
        if len(v) > MAX_PROMPT_CHARS:
            raise ValueError(f"Message too long (max {MAX_PROMPT_CHARS} chars)")
        return v

    @field_validator("max_tokens")
    @classmethod
    def cap_output(cls, v):
        if v < 1:
            raise ValueError("max_tokens must be positive")
        # При завышенном запросе уменьшаем лимит, а не отклоняем запрос.
        return min(v, MAX_OUTPUT_TOKENS)

class ChatResponse(BaseModel):
    response: str
    latency_ms: float

def check_rate_limit(request: Request):
    client_ip = request.client.host
    now = time.monotonic()
    # Проверяем и обновляем счётчик под одной блокировкой, иначе два потока
    # могут занять последнее место в окне одновременно.
    with _rate_lock:
        recent = [t for t in request_counts[client_ip] if now - t < RATE_WINDOW]
        if len(recent) >= RATE_LIMIT:
            request_counts[client_ip] = recent
            raise HTTPException(429, "Too many requests. Please wait a minute.")
        recent.append(now)
        request_counts[client_ip] = recent

def log_request(request_id, input_tokens, output_tokens, latency_ms, status):
    # Modal сохраняет stdout. Пишем только метаданные, без промптов,
    # ответов и исходного текста исключений.
    import json
    print(json.dumps({"ts": datetime.now().isoformat(),
                      "request_id": request_id,
                      "input_tokens": input_tokens,
                      "output_tokens": output_tokens,
                      "latency_ms": round(latency_ms, 2),
                      "status": status}))

# Управление сервером vLLM (один на контейнер)
_server = None
_tokenizer = None

def start_vllm():
    """Один раз запускаем vllm serve и ждём готовности."""
    global _server, _tokenizer
    if _server is None:
        _server = subprocess.Popen([
            "vllm", "serve", MODEL,
            "--host", "127.0.0.1",            # извне доступно только защищённое ASGI-приложение
            "--port", "8000",
            "--max-model-len", "8192",       # ограничиваем объём KV-кеша на L4
            "--gpu-memory-utilization", "0.90",
            "--dtype", "bfloat16",           # поддерживается на L4, но не на T4
        ])
        # Ждём готовности через /health самого vLLM, без генерации.
        for _ in range(120):
            try:
                requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
                break
            except Exception:
                time.sleep(2)
        else:
            raise RuntimeError("vLLM server did not become ready")
        from transformers import AutoTokenizer
        _tokenizer = AutoTokenizer.from_pretrained(MODEL)
    return _tokenizer

@web_app.get("/health")
def health():
    """Проверяем доступность локального vLLM без генерации."""
    try:
        requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
        return {"status": "healthy", "model": "Qwen3.5-2B"}
    except Exception:
        # Мониторинг должен видеть отказ: возвращаем 503, а не 200.
        return JSONResponse(status_code=503,
                            content={"status": "degraded", "model": "Qwen3.5-2B"})

@web_app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest, http_request: Request,
         _auth: None = Depends(require_token)):
    """Применяем шаблон чата и передаём подготовленный вход в vLLM."""
    check_rate_limit(http_request)
    request_id = uuid.uuid4().hex[:12]
    start = time.perf_counter()
    try:
        tokenizer = start_vllm()
        # Instruct-модели ожидают свой шаблон чата, а не сырой текст.
        # apply_chat_template оборачивает сообщение в специальные токены модели.
        messages = [{"role": "user", "content": request.message}]
        # Считаем токены после применения шаблона тем же токенизатором, что у vLLM.
        # Для одного диалога input_ids — плоский список. Длина всего словаря
        # дала бы число полей, а не токенов.
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True)["input_ids"]
        if len(input_ids) > MAX_INPUT_TOKENS:
            raise HTTPException(
                422, f"Message too long ({len(input_ids)} tokens; "
                     f"max {MAX_INPUT_TOKENS} input tokens)")
        # Передаём именно проверенные ID: /v1/completions принимает список
        # токенов в prompt, поэтому повторная токенизация не изменит бюджет.
        r = requests.post(
            "http://localhost:8000/v1/completions",
            json={"model": MODEL, "prompt": input_ids,
                  "max_tokens": request.max_tokens, "temperature": 0.7},
            timeout=120,
        )
        r.raise_for_status()
        body = r.json()
        text = body["choices"][0]["text"].strip()
        latency_ms = (time.perf_counter() - start) * 1000
        log_request(request_id, len(input_ids),
                    body.get("usage", {}).get("completion_tokens", 0),
                    latency_ms, 200)
        return ChatResponse(response=text, latency_ms=round(latency_ms, 2))
    except HTTPException as e:
        log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, e.status_code)
        raise
    except requests.HTTPError as e:
        # Ошибка vLLM: возвращаем клиенту тот же код 4xx, но без
        # исходного текста ошибки; сбой бэкенда отображаем в 502.
        status = e.response.status_code if e.response is not None else 502
        if 400 <= status < 500:
            log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, status)
            raise HTTPException(status, "The model rejected this request. "
                                        "Try a shorter message.")
        log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, 502)
        raise HTTPException(502, "Inference backend error. Please try again.")
    except Exception:
        log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, 500)
        raise HTTPException(500, "Generation failed. Please try again.")

@app.function(
    image=image,
    gpu="L4",
    timeout=600,
    scaledown_window=300,  # Не останавливать простаивающий контейнер 5 минут.
    volumes={"/root/.cache/huggingface": model_cache},  # Кеш между запусками.
    secrets=[modal.Secret.from_name("llm-api-token")],  # Предоставляет API_TOKEN.
)
@modal.concurrent(max_inputs=10)  # vLLM объединяет одновременные запросы в батчи.
@modal.asgi_app(requires_proxy_auth=True)
def serve():
    start_vllm()  # Загружаем модель при старте контейнера, а не при каждом запросе.
    return web_app

'''

print("Сохраните этот код как 'llm_service.py':")
print(llm_service_code)

In [ ]:
# Сохраните напечатанный код как check_service.py. Блокнот не выполняет запросы.
client_code = '''
import os
import requests

URL = os.environ["LLM_URL"].rstrip("/")  # точный HTTPS-адрес из вывода вашего развёртывания
if not URL.startswith("https://"):
    raise ValueError("LLM_URL must use HTTPS")
proxy = {"Modal-Key": os.environ["MODAL_KEY"], "Modal-Secret": os.environ["MODAL_SECRET"]}
chat = {**proxy, "Authorization": "Bearer " + os.environ["API_TOKEN"]}

# Сначала проверяем прокси. Этот запрос не должен запускать GPU.
r = requests.get(f"{URL}/health", timeout=30, allow_redirects=False)
assert r.status_code == 401, f"Expected proxy denial, got {r.status_code}"

r = requests.get(f"{URL}/health", headers=proxy, timeout=300, allow_redirects=False)
assert r.status_code == 200 and r.json()["status"] == "healthy", f"Readiness: {r.status_code}"

def ask(headers, payload):
    return requests.post(f"{URL}/chat", headers=headers, json=payload,
                         timeout=150, allow_redirects=False)

r = ask(proxy, {"message": "hi"})  # допуск на прокси не заменяет авторизацию в приложении
assert r.status_code == 401, f"Expected app denial, got {r.status_code}"
r = ask({**proxy, "Authorization": "Bearer wrong-token"}, {"message": "hi"})
assert r.status_code == 401, f"Expected wrong bearer denial, got {r.status_code}"
r = ask(chat, {"message": " "})
assert r.status_code == 422, f"Expected validation error, got {r.status_code}"
r = ask(chat, {"message": "What is Python?", "max_tokens": 64})
assert r.status_code == 200 and r.json()["response"].strip(), f"Chat: {r.status_code}"
print("Проверки прокси, готовности, токена приложения, валидации и ответа пройдены.")
'''
print(client_code)

### Контрольный список первого запуска

Эти шаги нужны только для **необязательной проверки в облаке**. Выполняйте их в своём терминале, а не в блокноте.

1. В панели Modal создайте в целевой среде секрет `llm-api-token` с ключом `API_TOKEN` и длинным случайным значением. Этот секрет нужен только GPU-сервису. Прокси-токен из §17.5 — отдельная пара. Учётные данные CLI не подходят ни для одного из этих уровней.
2. Сохраните напечатанные файлы `llm_service.py` и `check_service.py`. Настройте оповещение о расходах. Только если вы готовы оплачивать GPU, выполните `modal deploy llm_service.py`. Укажите в `LLM_URL` точный HTTPS-адрес из **собственного** вывода Modal, а не чужую ссылку. Передайте частному клиенту `API_TOKEN`, `MODAL_KEY` и `MODAL_SECRET` через менеджер секретов. Не вставляйте их в исходный код, вывод, URL или историю команд.
3. Запустите `python check_service.py`. Без заголовков ожидается 401 от прокси; с прокси-парой `/health` должен вернуть 200 и `status: healthy`. Ответ 503 **от приложения** означает, что проверка vLLM не прошла. Если готовность не восстанавливается, проверьте `modal app logs my-llm-service`. Не опрашивайте часто: даже без генерации допущенный запрос может запустить контейнер или задержать его остановку.
4. С одной прокси-парой (или неверным bearer-токеном) `/chat` должен вернуть 401 **от приложения**. С обоими уровнями доступа пустое сообщение должно дать 422, а корректное — 200 и непустое поле `response`. Таймаут функции Modal (600 с) не заменяет таймауты клиента. Клиент ограничивает ожидание каждого запроса и не следует перенаправлениям с секретными заголовками.
5. Даже если проверка не прошла, выполните `modal app stop my-llm-service` и убедитесь в панели управления, что работающих контейнеров не осталось. Отзовите демонстрационные учётные данные, когда они больше не нужны.

**Без развёртывания:** выполните локальные проверки главы 16 с подменой HTTP. Найдите в напечатанном клиенте запрос, который должен отклонять прокси до запуска контейнера, и запрос, который должно отклонять само приложение.

`/docs` и `/openapi.json` защищены тем же прокси. Заголовок `Authorization` на `/chat` предназначен только для токена приложения. `TestClient` вызывает FastAPI в обход прокси: так можно проверить авторизацию приложения, готовность с подменой vLLM и ограничения запросов, но не реальный допуск на границе Modal, готовность GPU или его стоимость.


## Раздел 17.7: Секреты и тома

### Создание секретов

В панели Modal создайте в среде функции секрет `my-secrets` с полем `API_KEY`. Ссылайтесь на него по имени; не записывайте значение в код и не выводите его. Прокси-пару хранит доверенный клиент, а не функция.

### Использование секретов в коде

In [ ]:
# Использование секретов в вашем коде
secrets_example = '''
@app.function(secrets=[modal.Secret.from_name("my-secrets")])
def with_secrets():
    import os
    api_key = os.environ["API_KEY"]
    # Безопасно используем ключ...
'''

print("Использование секретов:")
print(secrets_example)

In [ ]:
# Использование томов для кеширования моделей
volumes_example = '''
# Создаем том для кеша моделей
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)

@app.function(
    gpu="L4",
    volumes={"/root/.cache/huggingface": model_cache},
)
def with_cache():
    # Модели загружаются один раз, затем кешируются в томе
    pass
'''

print("Использование томов для кеширования моделей:")
print(volumes_example)
print("\nПервый запрос загружает модель. Последующие запросы используют кеш.")

## Раздел 17.8: Варианты GPU

| Размер модели | Рекомендуемый GPU | Стоимость/час |
|---------------|-------------------|---------------|
| < 3 млрд параметров (FP16/квантованные) | T4 (16GB) | $0.59 |
| < 3 млрд параметров (BF16, напр. Qwen3.5-2B) | L4 (24GB) | $0.80 |
| 3-8 млрд параметров | A10 (24GB) | $1.10 |
| 8-30 млрд параметров | A100-40GB | $2.50 |
| 30+ млрд параметров | H100 | $3.95 |

(Цены проверены 2026-11; уточняйте на modal.com/pricing.)

**Начните с L4.** Веса Qwen3.5-2B хранятся в BF16, а vLLM 0.34 отказывается запускать BF16 на GPU старше Ampere (compute capability 8.0): T4 — это 7.5, и запуск завершился бы ошибкой, а L4 (8.9) и A10 (8.6) проходят порог. T4 по-прежнему подходит для небольших FP16- или квантованных моделей. Обновляйте только при необходимости.

In [ ]:
# Примеры выбора GPU
print("Выбор GPU это один параметр:")
print()
print('@app.function(gpu="T4")      # 16GB, $0.59/час - только FP16 или квантованные модели (без BF16)')
print('@app.function(gpu="L4")      # 24GB, $0.80/час - наш выбор для Qwen3.5-2B')
print('@app.function(gpu="A10")     # 24GB, $1.10/час - та же память, быстрее вычисления')
print('@app.function(gpu="A100")    # Высокая производительность')
print('@app.function(gpu="H100")    # Максимальная мощность')
print('@app.function(gpu="A100:2")  # Два A100')

## Раздел 17.9: Управление развертыванием

### Просмотр логов

```bash
modal app logs my-llm-service
```

Или используйте панель управления на [modal.com](https://modal.com).

### Обновление приложения

```bash
# Просто повторно разверните
modal deploy llm_service.py
```

Без простоев. Modal автоматически обрабатывает непрерывные обновления.

### Контроль затрат

- Используйте `scaledown_window` для выключения простаивающих контейнеров
- Начните с L4, обновляйте только при необходимости
- Настройте оповещения о расходах в панели Modal; `modal app stop my-llm-service` полностью останавливает сервис
- Один запрос ограничен 4 000 символами, 2 048 входными токенами с шаблоном чата и 512 выходными токенами. Лимит в 20 запросов в минуту на IP действует под блокировкой **внутри одного контейнера** и сбрасывается при его остановке. Это не потолок общих расходов. Для нескольких пользователей и реплик нужны квоты в общем хранилище с атомарным обновлением; оповещение не останавливает расходы.

## Резюме

Вы научились:

1. **Развертывать функции** в Modal с помощью `@app.function()`
2. **Добавлять поддержку GPU** с помощью одного параметра: `gpu="L4"`
3. **Устанавливать зависимости** используя `modal.Image`
4. **Защищать все маршруты** до запуска контейнера с помощью `@modal.asgi_app(requires_proxy_auth=True)`
5. **Безопасно хранить секреты** с помощью `modal.Secret`
6. **Кешировать модели** с помощью `modal.Volume`
7. **Контролировать затраты** с помощью таймаутов простоя и мониторинга

**Делитесь интерфейсом, а не секретами.** У вас есть шаблон, а не подтверждение работы облачного сервиса. Только после успешной проверки развёртывания им могут пользоваться авторизованные клиенты. Прокси-пару храните на доверенном клиенте или сервере; отдельный bearer-токен нужен для генерации. Для конечных пользователей настройте собственный шлюз с авторизацией вместо раздачи учётных данных рабочей области.

**Команды для запоминания:**
```bash
pip install modal      # Установка
modal setup            # Аутентификация
modal run file.py      # Запустить один раз
modal deploy file.py   # Развернуть до явной остановки
modal app logs name    # Просмотр логов
modal app stop name    # Остановка приложения
```